# GREATER (biến thể XLM-R-Large) — GREedy Adversary PromoTed DefendeR

Li et al., **ACL 2025 Main Conference, Long Papers** — [`2025.acl-long.155`](https://aclanthology.org/2025.acl-long.155/),
DOI `10.18653/v1/2025.acl-long.155`, tr. 3091–3113. Code: [Liyuuuu111/GREATER](https://github.com/Liyuuuu111/GREATER).

Hai người chơi **cập nhật đồng bộ trong cùng một training step**:

| | Mục tiêu | Loss |
|---|---|---|
| **GREATER-A** (adversary) | Tìm từ quan trọng rồi thay để **đánh lừa** detector | `L_A = λ·L_adv + (1−λ)·L_imp` |
| **GREATER-D** (detector) | Dự đoán **đúng** trên cả text gốc lẫn text bị tấn công | `L_D = −log P(c\|X) − log P(c\|X̃)` |

`L_adv = −log P_tar(1−c \| X̃)` dùng **nhãn đảo** — adversary tối đa hoá xác suất nhãn sai, detector tối
thiểu hoá chính nó. Hai mục tiêu đối nghịch → **min-max thật**.

## Thay thế ở mức **TỪ**, không phải token-id

Cặp của chính paper là `M_sur` = RoBERTa-large (BPE, vocab 50,265) và `M_tar` = xlm-roberta-base
(SentencePiece, vocab 250,002) — **hai tokenizer khác hẳn**. Nếu thay ở mức token-id thì cặp đó không
chạy được. Figure 2 xác nhận là mức từ:

> gốc: *"…during the final stages of World **War** I … primarily composed of **British**, Canadian…"*
> tấn công: *"…during the final stages of World **Part** I … primarily composed of **China**, Canadian…"*

`Battle→Exam`, `War→Part`, `British→China` — **nguyên từ** bị thay trong văn bản, rồi `M_tar` tự
tokenize lại. Ràng buộc POS cũng nói *"candidate **words** must match the part of speech of the **words**
they replace"*.

→ `M_sur` **không cần chung vocab** với `M_tar`, và đó chính là điều cho phép giữ đúng threat model:
surrogate là model *thay thế* mà adversary dùng **vì không truy cập được target**.

## Quy trình tấn công (§4.1, verify từ PDF)

```
1. Identify   s_t = F_θ(h_t) trên token của M_sur → gộp về TỪ (max)
              I = top-k TỪ có điểm cao nhất        (k = 12% số từ, từ code gốc)
2. Perturb    δ⁰_t ~ U(-0.5, 0.5);  δ̂⁰_t = ξ·δ⁰_t/‖δ⁰_t‖₂
              δ_t = ε · ∇_δ̂ KL(P_sur(y|E) ‖ P_sur(y|Ẽ⁰)) / ‖∇…‖₂          (Eq. 5)
              ẽ_t = e_t + δ_t  chỉ với token thuộc từ trong I              (Eq. 4)
3. Project    C_t = top-m(Softmax(LMHead(M_sur(Ẽ))_t)) tại token ĐẦU của từ (m = 20)  (Eq. 6)
              → decode thành CHUỖI → lọc POS bằng nltk trên TỪ THẬT
4. Search     Algorithm 1 — thay lần lượt theo importance giảm dần, dựng lại TEXT,
              query M_tar; dừng khi lừa được HOẶC hết từ trong I
5. Prune      Algorithm 2 — hoàn nguyên từng từ; vẫn lừa được thì giữ bản gốc
```


> **Biến thể notebook này**: đổi `M_tar` sang **XLM-RoBERTa-large** (không phải `-base` như paper), **`M_sur` giữ nguyên `RoBERTa-large`** — CỐ ĐỊNH giống hệt `train_adv_01_greater_roberta.ipynb`, không đổi theo target. Đây là cách cô lập đúng biến target theo tinh thần ablation Table 3 của paper (đổi một biến, giữ biến kia cố định), cho phép so sánh chéo 2 biến thể target. `M_tar` đổi thành bản `-large` thay vì `-base` theo đúng Table 3, nơi `XLM-RoBERTa-Large` cho ASR thấp nhất (1.73%) sau khi train GREATER-D — robust nhất trong 7 backbone được test.

## Ba điểm quyết định tính đúng đắn

**1. Chỉ tấn công mặt AI.** §3: *"an adversary only maliciously edits the MGTs but would not make any
changes to the HWTs"*. Code gốc cũng chỉ sample từ `label == 1`.

**2. Chỉ tấn công 20% mẫu AI** (*"we use 20% of MGT for adversarial training"*) — đây là lý do notebook
này **nhẹ hơn RADAR**, không nặng hơn.

**3. Black-box thật.** `M_sur = roberta-large` **khác họ, khác vocab** với `M_tar = xlm-roberta-large`
— đúng cấu trúc paper. Không chỗ nào trong `greater_a` chạm vào nội trạng `M_tar`; nó chỉ gọi
`tar_predict(texts)` lấy output.


> **`SUR_MAX_LEN = 512`** (không phải 256) — khớp `MAX_LEN` của `M_tar`/supervised. Lý do: word_count trung bình dataset ~253-255 từ ⇒ ước tính ~330-350 token BPE, **vượt quá 256** ở phần lớn document. Nếu giữ 256, `word_view()`/`word_candidates()` cắt mất phần đuôi document khỏi tầm với của GREATER-A — không phải chỉ chậm hơn mà còn **thu hẹp phạm vi tấn công thật sự**, điều paper không hề đề cập. Để vừa VRAM mà không đổi số học attack: `PER_DEVICE_BATCH=4`, effective batch vẫn 32 nhờ accumulation; warm-up `M_sur` vẫn batch 8; xóa Adam state/gradient sau warm-up và **giữ gradient checkpointing** cho các forward cần gradient. `M_sur` trong GREATER-A vẫn FP32.

## Ba nguồn siêu tham số — ghi rõ từng cái

| | Paper (Table 6) | Code gốc | **Ta dùng** |
|---|---|---|---|
| λ / ε / ξ / U(a,b) | 0.05 / 0.3 / 0.01 / (−0.5, 0.5) | — | **y hệt paper** |
| Epoch / α label smoothing | 6 / 0.1 | 6 / 0.01 | **6 / 0.1** (paper) |
| LR detector | 1e-5 | 1e-6 | **1e-5** — trùng luôn supervised |
| Batch | 50 | 16 | **effective 32** (đồng bộ supervised) |
| top-k / top-m | *không nêu* | **12% / 20** | y hệt code |
| `F_θ` / LR / scheduler | *không nêu* | `Linear(h,256)→ReLU→Linear(256,1)→Sigmoid` / 1e-3 / `ExponentialLR(0.9)` | y hệt code |
| Surrogate | RoBERTa-large | GPT-2/HC3 | **RoBERTa-large** — khớp đúng paper (target đổi thành XLM-R-**Large**) |

## Đồng bộ supervised — KHÔNG đổi

`roberta-large` · `max_len` 512 · **effective batch 32** · LR 1e-5 · warmup 0.06 · wd 0.01 ·
`max_grad_norm` 1.0 · `adamw_bnb_8bit` · fp16 autocast · early-stop patience 2 theo AUROC(val) ·
`sample_weight` = 1.0 · **cùng block metric và cùng schema `pred_*.parquet`**.

In [ ]:
!pip install -q -U "transformers==4.57.6" datasets accelerate scikit-learn sentencepiece bitsandbytes nltk
import nltk
for p in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng", "universal_tagset"]:
    nltk.download(p, quiet=True)
print("san sang")

In [ ]:
%%writefile train_greater.py
# ============================================================================
#  GREATER bien the XLM-R-Large (Li et al., ACL 2025) -- 2 GPU tach vai tro, KHONG dung DDP.
#  M_tar = xlm-roberta-large (Table 3: ASR thap nhat sau GREATER-D) | M_sur = roberta-large (khop paper).
#  Chay : python train_greater.py     |  Smoke: SMOKE=1 python train_greater.py
#
#  Vi sao khong DDP: greedy search la vong lap TUAN TU (moi round query M_tar 1 lan).
#  Tach vai tro: M_tar tren cuda:0, M_sur + F_theta tren cuda:1.
#  Khong tensor nao co gradient di qua ranh gioi GPU -> khong NCCL, khong bay collective.
# ============================================================================
import os, json, math, random, time, warnings, gc
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import nltk
from sklearn.metrics import roc_auc_score, average_precision_score
from transformers import (AutoTokenizer, AutoModel, AutoModelForMaskedLM,
                          AutoModelForSequenceClassification, get_linear_schedule_with_warmup)

warnings.filterwarnings("ignore")

# ============================ CONFIG ========================================
RUN_NAME  = "greater_xlm_roberta_large"
DETECTOR  = "xlm-roberta-large"    # M_tar -- KHOP paper Table 3 (ASR thap nhat sau GREATER-D)
SURROGATE = "roberta-large"        # M_sur -- KHOP DUNG paper (RoBERTa-large lam surrogate); khac ho/vocab voi M_tar
SEED      = int(os.environ.get("RUN_SEED", "42"))
SMOKE     = os.environ.get("SMOKE", "0") == "1"

# ---- so cua PAPER (Table 6) ----
LAMBDA_A       = 0.05
EPS_SCALE      = 0.3
XI_SCALE       = 0.01
UNIF_A, UNIF_B = -0.5, 0.5
LABEL_SMOOTH   = 0.1
EPOCHS         = 6
ATTACK_FRAC    = 0.20     # "we use 20% of MGT for adversarial training"

# ---- so tu CODE goc (paper khong neu) ----
K_PERCENT   = 0.12        # top-k = 12% so TU cua moi text
TOP_M       = 20          # so ung vien
SCORE_LR    = 1e-3        # LR cua F_theta
SCHED_GAMMA = 0.9         # ExponentialLR moi epoch

# ---- DONG BO SUPERVISED (khop train_ddp.py cua supervised_models/) ----
MAX_LEN             = 512
EFFECTIVE_BATCH     = 32
PER_DEVICE_BATCH    = 4      # attack M_sur giu FP32; accum bu de effective batch van bang 32
SUR_WARMUP_BATCH    = 8      # giu nguyen batch warm-up M_sur de khong doi dong luc hoc surrogate
                             # (M_sur = roberta-large o CA HAI bien the -> warm-up phai giong het
                             #  thi ablation doi target moi co dung mot surrogate)
DET_LR              = 1e-5      # paper cung 1e-5
WARMUP_RATIO        = 0.06
WEIGHT_DECAY        = 0.01
ADAM_EPSILON        = 1e-8
MAX_GRAD_NORM       = 1.0
EARLY_STOP_PATIENCE = 2
TARGET_FPR          = 0.01
# ── Bản dataset dùng để train/eval ────────────────────────────────────
#   "deconf2"    = đã khử confound bề mặt v2 (02c_deconfound_v2.ipynb)  <- ĐANG DÙNG
#                  canonicalize + truncate_pair_sentences + lọc notation_gap 85%.
#                  Đo bằng shortcut_audit.py (22 đặc trưng, 4 tầng):
#                    tầng process (dấu vết quy trình) 0.5055  <- cổng cứng, PASS
#                    tổng 22 đặc trưng               0.8909
#                  7.025 cặp train (85% của 8.265) — lọc bỏ những cặp mà bản AI lược
#                  ký hiệu/số/tên riêng của bài gốc nhiều nhất.
#   "deconf"     = bản 02b_deconfound.ipynb. ĐỪNG dùng cho bản chính thức:
#                  02b tự báo "surface-GBM 0.6867, G6 PASS" nhưng con số đó chỉ đo 6
#                  đặc trưng, không có avg_word_len (0.7795) lẫn type_token_ratio
#                  (0.7727). Đo đủ 22: 0.9114. Và truncate_pair cắt thô theo từ làm
#                  ~nửa văn bản kết thúc giữa câu -> word_count 0.5000 + ends_terminal
#                  0.5236 khi đứng riêng nhưng GHÉP LẠI = 0.7264.
#   "normalized" = bản GỐC, CÒN shortcut: chỉ ĐẾM SỐ TỪ đã đoán đúng human/AI
#                  ở AUROC 0.7193 (surface-GBM 0.8080). G6 FAIL.
#
# PHẢI đặt TRONG train_ddp.py, KHÔNG phải trong notebook: file này chạy qua
# torchrun ở tiến trình RIÊNG nên biến của notebook không tới được. Đặt ngoài thì
# script chết NameError — và trước đó cell còn không chạy nổi, vì dòng
# `DATASET_TAG = ...` ở trên đẩy %%writefile khỏi dòng đầu, làm Jupyter hiểu nó
# thành line magic: "Line magic function `%%writefile` not found".
#
# LƯU Ý: Kaggle dataset phải chứa dataset_{train,val,test_ood_model,test_fp}_deconf2.parquet
#        VÀ matched_eval_{val,test_ood_model}_deconf2.parquet (đánh giá đối-shortcut).
#        Danh sách file cần upload: KAGGLE_UPLOAD.md ở gốc repo.
DATASET_TAG = "deconf2"

SURROGATE_EPOCHS = 1
SUR_MAX_LEN      = 512    # khop M_tar/supervised -- tranh cat mat duoi document khoi tam voi tan cong (xem markdown)
MIN_WORD_LEN     = 3
GC_EVERY         = 200   # gc.collect() dinh ky: do thi autograd tao reference cycle,
                         # Python chi thu vong bang generational GC (trigger theo SO LUONG
                         # object, khong theo ap luc VRAM) -> tensor CUDA ket trong cycle
                         # khong duoc tra ve cho toi khi gc chay.
PROBE_ITERS      = int(os.environ.get("PROBE_ITERS", "100"))
KAGGLE_INPUT     = "/kaggle/input/datasets/<kaggle-user>/halfdatadetector"

if SMOKE:
    EPOCHS, SURROGATE_EPOCHS, ATTACK_FRAC = 1, 1, 0.5

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
grad_accum = max(1, EFFECTIVE_BATCH // PER_DEVICE_BATCH)
assert PER_DEVICE_BATCH * grad_accum == EFFECTIVE_BATCH

DET_DEV = "cuda:0" if torch.cuda.is_available() else "cpu"
SUR_DEV = "cuda:1" if torch.cuda.device_count() > 1 else DET_DEV
print("M_tar(%s) -> %s | M_sur(%s) + F_theta -> %s" % (DETECTOR, DET_DEV, SURROGATE, SUR_DEV), flush=True)
print("per_device=%d x accum=%d = effective %d (ky vong 32)" % (PER_DEVICE_BATCH, grad_accum, EFFECTIVE_BATCH), flush=True)

# ============================ DATA ==========================================
def find_data_dir():
    t = f"dataset_train_{DATASET_TAG}.parquet"
    for b in [Path(KAGGLE_INPUT), Path("output"), Path("."), Path("/kaggle/input")]:
        if b.exists():
            if (b / t).exists():
                return b
            h = list(b.rglob(t))
            if h:
                return h[0].parent
    raise FileNotFoundError(t)

DATA_DIR = find_data_dir()
OUT_DIR = (Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")) / \
          ("artifacts_%s_seed%d" % (RUN_NAME, SEED))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR:", DATA_DIR, "| OUT_DIR:", OUT_DIR, flush=True)

def melt_pairs(df):
    """sample_weight BO (plan §4.1) -> khong mang sang."""
    h = pd.DataFrame({"text": df["human_text"], "label": 0, "paper_id": df["paper_id"],
                      "ai_degree": 0, "generator_family": "human"})
    a = pd.DataFrame({"text": df["ai_text"], "label": 1, "paper_id": df["paper_id"],
                      "ai_degree": df.get("ai_degree", 3),
                      "generator_family": df.get("generator_family", "unknown")})
    o = pd.concat([h, a], ignore_index=True)
    return o[o["text"].notna() & (o["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def melt_fp(df):
    has_ai = df["ai_text"].notna() if "ai_text" in df.columns else pd.Series(False, index=df.index)
    text = df["ai_text"].where(has_ai, df["human_text"]) if "ai_text" in df.columns else df["human_text"]
    o = pd.DataFrame({"text": text, "label": 0, "paper_id": df["paper_id"],
                      "ai_degree": df["ai_degree"].fillna(0).astype(int) if "ai_degree" in df.columns else 0,
                      "generator_family": "human"})
    return o[o["text"].notna() & (o["text"].astype(str).str.len() > 0)].reset_index(drop=True)

def load(n, fp=False):
    p = DATA_DIR / ("dataset_%s_%s.parquet" % (n, DATASET_TAG))
    if not p.exists():
        print("  thieu %s -- bo qua" % p.name); return None
    df = pd.read_parquet(p)
    return melt_fp(df) if fp else melt_pairs(df)

train_df = load("train"); val_df = load("val")
ood_df = load("test_ood_model"); fp_df = load("test_fp", fp=True)
if SMOKE:
    train_df = train_df.sample(n=min(256, len(train_df)), random_state=SEED).reset_index(drop=True)
    val_df = val_df.groupby("label", group_keys=False).sample(n=64, random_state=SEED).reset_index(drop=True)
    ood_df = None; fp_df = None
for nm, d in [("train", train_df), ("val", val_df), ("ood", ood_df), ("test_fp", fp_df)]:
    if d is not None:
        print("%-10s %7s mau  label=%s" % (nm, format(len(d), ","), d["label"].value_counts().to_dict()), flush=True)

# ============================ MODELS ========================================
tok_tar = AutoTokenizer.from_pretrained(DETECTOR)
tok_sur = AutoTokenizer.from_pretrained(SURROGATE)
assert tok_sur.is_fast, "can fast tokenizer de lay offset_mapping/word_ids"

M_tar = AutoModelForSequenceClassification.from_pretrained(DETECTOR, num_labels=2).to(DET_DEV)
M_tar.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
M_tar.config.use_cache = False

class Surrogate(nn.Module):
    """M_sur dung 1 encoder cho HAI viec:
       (a) lm_head  -> chieu embedding nhieu ve vocab (Eq. 6)
       (b) cls head -> P_sur(y|E) cho KL (Eq. 5) va gradient s* (Eq. 8)"""
    def __init__(self, name):
        super().__init__()
        self.mlm = AutoModelForMaskedLM.from_pretrained(name)
        self.enc = self.mlm.base_model               # xlm-roberta / roberta backbone
        h = self.mlm.config.hidden_size
        self.cls = nn.Linear(h, 2)
        self.hidden_size = h

    def embed(self, input_ids):
        return self.enc.embeddings.word_embeddings(input_ids)

    def forward(self, input_ids=None, inputs_embeds=None, attention_mask=None):
        hs = self.enc(input_ids=input_ids, inputs_embeds=inputs_embeds,
                      attention_mask=attention_mask).last_hidden_state
        return {"hidden": hs, "logits": self.cls(hs[:, 0])}

    def lm_head(self, hs):
        return self.mlm.lm_head(hs)

M_sur = Surrogate(SURROGATE).to(SUR_DEV)
F_theta = nn.Sequential(nn.Linear(M_sur.hidden_size, 256), nn.ReLU(),
                        nn.Linear(256, 1), nn.Sigmoid()).to(SUR_DEV)   # kien truc code goc

try:
    import bitsandbytes as bnb
    det_opt = bnb.optim.AdamW8bit(M_tar.parameters(), lr=DET_LR,
                                  weight_decay=WEIGHT_DECAY, eps=ADAM_EPSILON)
    print("optimizer detector: adamw_bnb_8bit (dong bo supervised)", flush=True)
except Exception as e:
    det_opt = torch.optim.AdamW(M_tar.parameters(), lr=DET_LR,
                                weight_decay=WEIGHT_DECAY, eps=ADAM_EPSILON)
    print("optimizer detector: AdamW fp32 (%s)" % e, flush=True)
adv_opt = torch.optim.AdamW(F_theta.parameters(), lr=SCORE_LR)

steps_per_epoch = max(1, math.ceil(len(train_df) / EFFECTIVE_BATCH))
det_sched = get_linear_schedule_with_warmup(det_opt, int(WARMUP_RATIO * steps_per_epoch * EPOCHS),
                                            steps_per_epoch * EPOCHS)
adv_sched = torch.optim.lr_scheduler.ExponentialLR(adv_opt, gamma=SCHED_GAMMA)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
AMP = dict(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available())

# ============================ WARM-UP M_sur =================================
# Paper: "a TRAINED RoBERTa Large as the Surrogate Model" -> M_sur phai la detector da train
# thi gradient s* (Eq. 8) va KL (Eq. 5) moi co nghia.
print("\n--- warm-up M_sur = %s (%d epoch, max_len=%d) ---" % (SURROGATE, SURROGATE_EPOCHS, SUR_MAX_LEN), flush=True)
sur_opt = torch.optim.AdamW(M_sur.parameters(), lr=2e-5)
sur_scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
M_sur.enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
M_sur.train(); t_w = time.time()
for ep in range(SURROGATE_EPOCHS):
    idx = np.random.RandomState(SEED + ep).permutation(len(train_df))
    for i in range(0, len(idx), SUR_WARMUP_BATCH):
        b = train_df.iloc[idx[i:i + SUR_WARMUP_BATCH]]
        enc = tok_sur(b["text"].tolist(), truncation=True, max_length=SUR_MAX_LEN,
                      padding=True, return_tensors="pt").to(SUR_DEV)
        y = torch.tensor(b["label"].values, device=SUR_DEV)
        with torch.autocast(**AMP):
            loss = F.cross_entropy(M_sur(**enc)["logits"].float(), y)
        sur_opt.zero_grad(set_to_none=True)
        sur_scaler.scale(loss).backward(); sur_scaler.unscale_(sur_opt)
        torch.nn.utils.clip_grad_norm_(M_sur.parameters(), 1.0)
        sur_scaler.step(sur_opt); sur_scaler.update()
        if (i // SUR_WARMUP_BATCH) % 100 == 0:
            print("   ep%d step%-5d loss %.4f  (%.1f min)"
                  % (ep, i // SUR_WARMUP_BATCH, float(loss), (time.time() - t_w) / 60), flush=True)
# BUG DA SUA: truoc day goi gradient_checkpointing_disable() + M_sur.eval() o day. HF chi bat
# gradient checkpointing khi "self.gradient_checkpointing and self.training"
# (transformers/modeling_layers.py, GenericLayerMixin.__call__) -> M_sur.eval() mot minh cung du
# TAT NGAM checkpointing cho toan bo GREATER-A ve sau, du gradient_checkpointing_enable() da goi.
# He qua: moi forward qua M_sur trong word_candidates()/adversary_loss() tich day activation FP32
# KHONG checkpoint -> VRAM GPU1 (SUR_DEV) nam sat tran suot nhieu epoch, roi mot micro-batch hoi
# nang hon (text dai hon / nhieu tu bi chon hon) lam OOM giua chung.
# Fix: dua dropout ve 0 (XLM-R/RoBERTa dung LayerNorm, khong co running-stats nen train/eval khong
# con khac biet gi khac) roi GIU train() -> checkpointing hoat dong dung nhu y dinh ban dau.
for m in M_sur.modules():
    if isinstance(m, nn.Dropout):
        m.p = 0.0
M_sur.train()
for p in M_sur.parameters():
    p.requires_grad_(False)          # dong bang: trong vong doi khang CHI F_theta hoc
    p.grad = None                    # requires_grad_(False) tu no KHONG giai phong gradient cu

# AdamW warm-up giu 2 moment FP32 cho toan bo RoBERTa-large (~vai GB) neu khong xoa.
# Giai phong truoc GREATER-A; attack van chay FP32 de bao toan tinh cong bang/phuong phap.
del sur_opt, sur_scaler, loss, enc, y, b
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("da giai phong optimizer/gradient warm-up M_sur", flush=True)

# ============================ WORD VIEW =====================================
def word_view(texts):
    """Tokenize bang M_sur roi GOP subword -> TU.
    Tra ve: enc (tensors), spans[i] = list[(char_start, char_end)] cua tung tu,
            w_of_tok[i] = list[word_index | -1] cho tung token."""
    enc = tok_sur(list(texts), truncation=True, max_length=SUR_MAX_LEN, padding=True,
                  return_offsets_mapping=True, return_tensors="pt")
    spans, w_of_tok = [], []
    for i in range(len(texts)):
        wid = enc.word_ids(i)
        off = enc["offset_mapping"][i].tolist()
        sp, cur = {}, []
        for t, w in enumerate(wid):
            if w is None or off[t][1] <= off[t][0]:
                cur.append(-1); continue
            cur.append(w)
            s, e = off[t]
            if w in sp:
                sp[w] = (min(sp[w][0], s), max(sp[w][1], e))
            else:
                sp[w] = (s, e)
        keys = sorted(sp)
        remap = {k: j for j, k in enumerate(keys)}
        spans.append([sp[k] for k in keys])
        w_of_tok.append([remap.get(w, -1) if w is not None else -1 for w in cur])
    enc.pop("offset_mapping")
    return enc, spans, w_of_tok

def render(text, spans, repl):
    """Dung lai TEXT voi cac tu da thay. Duyet TU CUOI VE DAU -> offset goc con dung."""
    out = text
    for wi in sorted(repl, reverse=True):
        s, e = spans[wi]
        out = out[:s] + repl[wi] + out[e:]
    return out

_pos = {}
def pos_of(w):
    w = w.strip()
    if not w:
        return None
    if w not in _pos:
        try:
            _pos[w] = nltk.pos_tag([w], tagset="universal")[0][1]
        except Exception:
            _pos[w] = None
    return _pos[w]

@torch.no_grad()
def tar_predict(texts, batch=16):
    """BLACK-BOX: chi lay OUTPUT cua M_tar tren TEXT. Khong cham noi trang."""
    was_training = M_tar.training
    M_tar.eval(); preds, probs = [], []
    try:
        for i in range(0, len(texts), batch):
            enc = tok_tar(texts[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True,
                          pad_to_multiple_of=8, return_tensors="pt").to(DET_DEV)
            with torch.autocast(**AMP):
                lg = M_tar(**enc).logits
            p = torch.softmax(lg.float(), -1)
            preds.append(p.argmax(-1).cpu()); probs.append(p.cpu())
    finally:
        M_tar.train(was_training)   # query black-box khong duoc lam detector ket o eval mode
    return torch.cat(preds), torch.cat(probs)

# ============================ GREATER-A =====================================
def word_candidates(enc, spans, w_of_tok, texts):
    """Eq. 2-6: diem importance theo TU, nhieu co huong, chieu ve vocab, loc POS.
    Tra ve: order[i] = list word_idx theo importance giam dan (da cat top-k),
            cands[i] = {word_idx: [chuoi ung vien]}"""
    ids = enc["input_ids"].to(SUR_DEV); attn = enc["attention_mask"].to(SUR_DEV)
    B, T = ids.shape

    # BUG DA SUA: s_tok truoc day nam NGOAI no_grad. F_theta co requires_grad=True nen dong nay
    # dung mot do thi autograd giu h [B,512,1024] fp32 + intermediate F_theta, roi vong cham diem
    # ben duoi (sc[w] = maximum(...)) them ~512*B node in-place moi step. Gradient do KHONG BAO GIO
    # duoc dung: s_tok chi di toi topk().indices.tolist() de XEP HANG tu; F_theta hoc o
    # adversary_loss qua duong L_imp hoan toan khac. Do thi thua nay la nguon ro VRAM tren SUR_DEV.
    with torch.no_grad():
        h = M_sur(input_ids=ids, attention_mask=attn)["hidden"]
        s_tok = F_theta(h).squeeze(-1)                                    # Eq. 2
        del h                        # chi dung cho s_tok, khong tai su dung ben duoi

    # --- gop diem token -> TU (max), chon top-k TU ---
    wt = torch.tensor([r + [-1] * (T - len(r)) for r in w_of_tok], device=SUR_DEV)[:, :T]
    order, keep_mask = [], torch.zeros(B, T, device=SUR_DEV)
    for i in range(B):
        nw = len(spans[i])
        if nw == 0:
            order.append([]); continue
        sc = torch.full((nw,), -1e4, device=SUR_DEV)
        for t in range(T):
            w = int(wt[i, t])
            if w >= 0:
                sc[w] = torch.maximum(sc[w], s_tok[i, t])
        k = max(1, int(round(K_PERCENT * nw)))                            # Eq. 3, k = 12% so tu
        top = torch.topk(sc, k=min(k, nw)).indices.tolist()
        order.append(top)
        sel = set(top)
        for t in range(T):
            if int(wt[i, t]) in sel:
                keep_mask[i, t] = 1.0

    # --- Eq. 4-5: nhieu co huong tren embedding cua token thuoc tu duoc chon ---
    E = M_sur.embed(ids)
    with torch.no_grad():
        p_orig = F.softmax(M_sur(inputs_embeds=E, attention_mask=attn)["logits"], -1)
    d0 = torch.empty_like(E).uniform_(UNIF_A, UNIF_B)
    d0 = XI_SCALE * d0 / (d0.norm(dim=-1, keepdim=True) + 1e-12)
    d0 = (d0 * keep_mask.unsqueeze(-1)).detach().requires_grad_(True)
    p_pert = F.log_softmax(M_sur(inputs_embeds=E + d0, attention_mask=attn)["logits"], -1)
    kl = F.kl_div(p_pert, p_orig, reduction="batchmean")
    g = torch.autograd.grad(kl, d0)[0]
    delta = EPS_SCALE * g / (g.norm(dim=-1, keepdim=True) + 1e-12)        # Eq. 5

    # --- Eq. 6: chieu ve vocab CHI tai token DAU cua moi tu duoc chon ---
    # (chi tai I -> giam ~10x bo nho so voi chieu ca chuoi: vocab 250k x 512 x B la rat lon)
    with torch.no_grad():
        hs = M_sur(inputs_embeds=E + delta * keep_mask.unsqueeze(-1), attention_mask=attn)["hidden"]
    head_pos = []                                     # (i, t, word_idx) token dau cua tung tu chon
    for i in range(B):
        seen = set()
        for t in range(T):
            w = int(wt[i, t])
            if w >= 0 and w in set(order[i]) and w not in seen:
                seen.add(w); head_pos.append((i, t, w))
    cands = [dict() for _ in range(B)]
    if head_pos:
        bi = torch.tensor([p[0] for p in head_pos], device=SUR_DEV)
        ti = torch.tensor([p[1] for p in head_pos], device=SUR_DEV)
        with torch.no_grad():
            lm = M_sur.lm_head(hs[bi, ti])                                 # [N, vocab]
            top_ids = lm.topk(TOP_M, dim=-1).indices.cpu().tolist()
        for (i, _, w), ids_m in zip(head_pos, top_ids):
            s, e = spans[i][w]
            orig_w = texts[i][s:e]
            if len(orig_w.strip()) < MIN_WORD_LEN:
                continue
            tag = pos_of(orig_w)
            out = []
            for cid in ids_m:                                              # loc POS tren TU THAT
                cw = tok_sur.decode([cid]).strip()
                if (not cw) or (not cw.isascii()) or len(cw) < MIN_WORD_LEN:
                    continue
                if cw.lower() == orig_w.strip().lower():
                    continue
                if tag is not None and pos_of(cw) != tag:
                    continue
                out.append(cw)
            if out:
                cands[i][w] = out
        del bi, ti, lm, top_ids
    # tra VRAM ngay thay vi doi refcount: ham nay bi goi ~1500 lan/epoch
    del ids, attn, E, d0, p_orig, p_pert, kl, g, delta, hs, s_tok, wt, keep_mask
    return order, cands

def greater_a(texts, labels):
    """Algorithm 1 (greedy search) + Algorithm 2 (greedy pruning), chay THEO LO:
    ca sub-batch cung tien 1 round -> 1 lan query M_tar cho ca lo."""
    texts = list(texts)
    enc, spans, w_of_tok = word_view(texts)
    order, cands = word_candidates(enc, spans, w_of_tok, texts)

    B = len(texts)
    repl = [dict() for _ in range(B)]
    done = torch.zeros(B, dtype=torch.bool)
    y = labels.detach().cpu()
    rounds = max((len(o) for o in order), default=0)

    for r in range(rounds):                                  # --- Algorithm 1 ---
        touched = False
        for i in range(B):
            if done[i] or r >= len(order[i]):
                continue
            w = order[i][r]
            if w in cands[i]:
                repl[i][w] = cands[i][w][0]                  # ung vien xac suat cao nhat
                touched = True
        if not touched:
            continue
        cur = [render(texts[i], spans[i], repl[i]) for i in range(B)]
        pred, _ = tar_predict(cur)
        done |= (pred != y)                                  # lua duoc -> dung
        if done.all():
            break

    for r in range(rounds):                                  # --- Algorithm 2: pruning ---
        act = [i for i in range(B) if r < len(order[i]) and order[i][r] in repl[i]]
        if not act:
            continue
        trial = []
        for i in range(B):
            rp = dict(repl[i])
            if i in act:
                rp.pop(order[i][r], None)                    # hoan nguyen thu
            trial.append(render(texts[i], spans[i], rp))
        pred, _ = tar_predict(trial)
        for i in act:
            if bool(pred[i] != y[i]):                        # van lua duoc -> giu ban goc
                repl[i].pop(order[i][r], None)

    adv = [render(texts[i], spans[i], repl[i]) for i in range(B)]
    n_ch = sum(len(r) for r in repl)
    return adv, n_ch, float(done.float().mean()), enc, spans, w_of_tok

# ============================ LOSSES ========================================
def adversary_loss(enc, labels, adv_texts):
    """L_A = lambda*L_adv + (1-lambda)*L_imp  (Eq. 8-11).

    LUU Y: X~ la TEXT ROI RAC nen L_adv KHONG co duong gradient toi F_theta.
    No vao tong loss nhu HANG SO (chi de log/can bang ty le); gradient thuc te den tu L_imp
    -- khop voi lambda=0.05 rat nho va voi mo ta cua code goc ("compute generator loss
    against proxy model gradients"). Ghi ro trong bang Deviation."""
    ids = enc["input_ids"].to(SUR_DEV); attn = enc["attention_mask"].to(SUR_DEV)
    y = labels.to(SUR_DEV)

    E = M_sur.embed(ids).detach().requires_grad_(True)
    L_sur = F.cross_entropy(M_sur(inputs_embeds=E, attention_mask=attn)["logits"], y)
    s_star = torch.autograd.grad(L_sur, E)[0].norm(dim=-1).detach()        # Eq. 8
    s_star = s_star / (s_star.amax(dim=1, keepdim=True) + 1e-12)           # ve [0,1] cho khop sigmoid

    h = M_sur(input_ids=ids, attention_mask=attn)["hidden"].detach()
    s = F_theta(h).squeeze(-1)
    m = attn.float()
    L_imp = (((s - s_star) ** 2) * m).sum() / m.sum().clamp(min=1)         # Eq. 9

    _, prob = tar_predict(adv_texts)                                        # Eq. 10, nhan DAO
    p_wrong = prob.gather(1, (1 - labels.cpu()).unsqueeze(1)).squeeze(1)
    L_adv = float(-(p_wrong.clamp_min(1e-8).log()).mean())

    out = (1 - LAMBDA_A) * L_imp, L_adv, float(L_imp)
    del ids, attn, E, L_sur, s_star, h, s, m, L_imp
    return out

def detector_loss(texts, y, adv_texts, has_adv):
    """L_D = -log P(c|X) - log P(c|X~)  (Eq. 12), label smoothing alpha=0.1."""
    enc = tok_tar(list(texts), truncation=True, max_length=MAX_LEN, padding=True,
                  pad_to_multiple_of=8, return_tensors="pt").to(DET_DEV)
    yd = y.to(DET_DEV)
    with torch.autocast(**AMP):
        loss = F.cross_entropy(M_tar(**enc).logits.float(), yd, label_smoothing=LABEL_SMOOTH)
        if has_adv:
            enc_a = tok_tar(list(adv_texts), truncation=True, max_length=MAX_LEN, padding=True,
                            pad_to_multiple_of=8, return_tensors="pt").to(DET_DEV)
            loss = loss + F.cross_entropy(M_tar(**enc_a).logits.float(), yd,
                                          label_smoothing=LABEL_SMOOTH)
    return loss

# ============================ EVAL ==========================================
@torch.no_grad()
def predict_proba(texts, batch=16):
    """Khop y het predict_proba() cua supervised (train_ddp.py): KHONG bao autocast fp16,
    chay fp32 thuan sau khi load lai model -- dam bao eval nhat quan 1:1 giua 2 nhanh."""
    M_tar.eval(); out = []
    texts = list(texts)
    for i in range(0, len(texts), batch):
        enc = tok_tar(texts[i:i + batch], truncation=True, max_length=MAX_LEN, padding=True,
                      pad_to_multiple_of=8, return_tensors="pt").to(DET_DEV)
        lg = M_tar(**enc).logits
        out.extend(torch.softmax(lg.float(), -1)[:, 1].cpu().tolist())
    return np.array(out)

def val_auroc():
    return float(roc_auc_score(val_df["label"].values, predict_proba(val_df["text"].tolist())))

# ============================ PROBE VRAM ====================================
# Chay: SMOKE=1 PROBE=1 python train_greater.py   (~2 phut; SMOKE lam warm-up chi con 32 step)
# Muc dich: xac nhan duong tan cong khong con ro VRAM TRUOC khi cam ket ~8 tieng full run.
# Doc ket qua:
#   "delta vs base" di ngang quanh 0        -> sach, chay full duoc
#   "delta vs base" leo deu theo vong       -> van con leak, dung chay full
#   "alloc" cao nhung "sau gc" tut manh     -> leak do reference cycle, ha GC_EVERY xuong
if os.environ.get("PROBE", "0") == "1" and torch.cuda.is_available():
    print("\n" + "=" * 78)
    print("PROBE VRAM tren %s -- %d vong tren CUNG mot batch" % (SUR_DEV, PROBE_ITERS))
    print("=" * 78, flush=True)
    _ai = train_df[train_df["label"] == 1].head(PER_DEVICE_BATCH)
    _t = _ai["text"].tolist(); _y = torch.tensor(_ai["label"].values)
    _mb = lambda: torch.cuda.memory_allocated(SUR_DEV) / 2 ** 20
    gc.collect(); torch.cuda.reset_peak_memory_stats(SUR_DEV)
    _base = _mb()
    print("  base (chi M_sur + F_theta): %.0f MB" % _base, flush=True)
    for _it in range(PROBE_ITERS):
        _a, _, _, _e, _, _ = greater_a(_t, _y)
        _la, _, _ = adversary_loss(_e, _y, _a)
        adv_opt.zero_grad(set_to_none=True); _la.backward(); adv_opt.step()
        del _a, _e, _la
        if (_it + 1) % 10 == 0:
            _a1 = _mb(); gc.collect(); _a2 = _mb()
            print("  vong %3d | alloc %7.0f MB | sau gc %7.0f MB | delta vs base %+7.0f MB"
                  " | peak %7.0f MB"
                  % (_it + 1, _a1, _a2, _a2 - _base,
                     torch.cuda.max_memory_allocated(SUR_DEV) / 2 ** 20), flush=True)
    print("\nKET LUAN: 'delta vs base' phai gan 0 va KHONG tang dan theo vong.", flush=True)
    raise SystemExit(0)

# ============================ TRAINING LOOP =================================
print("\n" + "=" * 78); print("ADVERSARIAL TRAINING -- %d epoch" % EPOCHS); print("=" * 78, flush=True)
best, bad, monitor = -1.0, 0, []
rs = np.random.RandomState(SEED)

for ep in range(EPOCHS):
    idx = rs.permutation(len(train_df))
    M_tar.train(); det_opt.zero_grad(set_to_none=True)
    t0 = time.time(); agg = []
    for i in range(0, len(idx), PER_DEVICE_BATCH):
        b = train_df.iloc[idx[i:i + PER_DEVICE_BATCH]]
        texts = b["text"].tolist()
        y = torch.tensor(b["label"].values)

        # --- chon tap bi tan cong: CHI mat AI, va chi ATTACK_FRAC trong so do ---
        pick = [j for j in range(len(b)) if y[j] == 1 and rs.rand() < ATTACK_FRAC]
        has_adv = len(pick) > 0
        adv_texts = list(texts)
        n_ch = 0; asr = float("nan"); l_adv = l_imp = float("nan")

        if has_adv:
            sub_t = [texts[j] for j in pick]
            sub_y = y[pick]
            a_texts, n_ch, asr, enc_s, _, _ = greater_a(sub_t, sub_y)
            for j, t in zip(pick, a_texts):
                adv_texts[j] = t
            # (1) cap nhat ADVERSARY (F_theta), M_tar dong bang
            la, l_adv, l_imp = adversary_loss(enc_s, sub_y, a_texts)
            adv_opt.zero_grad(set_to_none=True); la.backward()
            torch.nn.utils.clip_grad_norm_(F_theta.parameters(), MAX_GRAD_NORM); adv_opt.step()

        # (2) cap nhat DETECTOR trong CUNG step
        ld = detector_loss(texts, y, adv_texts, has_adv) / grad_accum
        scaler.scale(ld).backward()
        if (i // PER_DEVICE_BATCH + 1) % grad_accum == 0:
            scaler.unscale_(det_opt)
            torch.nn.utils.clip_grad_norm_(M_tar.parameters(), MAX_GRAD_NORM)
            scaler.step(det_opt); scaler.update(); det_sched.step()
            det_opt.zero_grad(set_to_none=True)

        agg.append((float(ld) * grad_accum, l_adv, l_imp, asr, n_ch))
        step = i // PER_DEVICE_BATCH
        if step % GC_EVERY == 0:
            gc.collect()             # cat reference cycle giu tensor CUDA (xem GC_EVERY)
        if step % 50 == 0:
            vram = ""
            if torch.cuda.is_available():
                # vram phai DI NGANG. Leo deu qua nhieu epoch = con ro -> chay PROBE de khoanh vung.
                vram = " | vram sur %.1fG tar %.1fG" % (
                    torch.cuda.memory_allocated(SUR_DEV) / 2 ** 30,
                    torch.cuda.memory_allocated(DET_DEV) / 2 ** 30)
            print("  ep%d %5d/%d | L_D %.4f | L_adv %.4f | L_imp %.4f | ASR %.2f | doi %d tu | %.1f min%s"
                  % (ep, i, len(idx), float(ld) * grad_accum, l_adv, l_imp, asr, n_ch,
                     (time.time() - t0) / 60, vram), flush=True)

    adv_sched.step()
    a = val_auroc()
    A = np.array([[x if x == x else np.nan for x in row] for row in agg], dtype=float)
    monitor.append({"epoch": ep, "val_auroc": a, "L_D": float(np.nanmean(A[:, 0])),
                    "L_adv": float(np.nanmean(A[:, 1])), "L_imp": float(np.nanmean(A[:, 2])),
                    "ASR": float(np.nanmean(A[:, 3])), "words_changed": float(np.nansum(A[:, 4])),
                    "minutes": (time.time() - t0) / 60})
    pd.DataFrame(monitor).to_csv(OUT_DIR / "monitor.csv", index=False)   # ghi MOI epoch:
    # truoc day chi ghi sau vong lap nen mot lan crash o epoch 4 lam mat sach monitor 4 epoch.
    print("  -> epoch %d: AUROC(val) = %.4f | ASR tb = %.3f" % (ep, a, monitor[-1]["ASR"]), flush=True)
    if a > best:
        best, bad = a, 0
        if not SMOKE:
            M_tar.save_pretrained(OUT_DIR / "model"); tok_tar.save_pretrained(OUT_DIR / "model")
            torch.save(F_theta.state_dict(), OUT_DIR / "scoring_network.pt")
            print("     phi_best MOI -> da luu", flush=True)
    else:
        bad += 1
        if bad >= EARLY_STOP_PATIENCE:
            print("     EARLY STOP", flush=True); break

pd.DataFrame(monitor).to_csv(OUT_DIR / "monitor.csv", index=False)
if SMOKE:
    print("\nSMOKE xong -- kiem cac gate o markdown cell."); raise SystemExit(0)

# ============================ EVAL CUOI =====================================
print("\n" + "=" * 78); print("EVAL phi_best"); print("=" * 78, flush=True)
# det_opt/det_sched con giu parameter va Adam state cua model cu. Phai xoa TRUOC
# khi load phi_best, neu khong GPU 0 tam thoi chua ca model cu lan model moi.
del det_opt, det_sched, scaler, M_tar
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
M_tar = AutoModelForSequenceClassification.from_pretrained(OUT_DIR / "model").to(DET_DEV).eval()
for d in [val_df, ood_df, fp_df]:
    if d is not None:
        d["score"] = predict_proba(d["text"].tolist())

def tpr_at_fpr(y, s, fpr):
    y, s = np.asarray(y), np.asarray(s); neg = s[y == 0]
    if len(neg) == 0 or (y == 1).sum() == 0:
        return np.nan, np.nan
    thr = np.quantile(neg, 1 - fpr, method="higher")
    return float((s[y == 1] >= thr).mean()), float(thr)

def auroc(y, s):
    y = np.asarray(y)
    return float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else np.nan

_, TAU = tpr_at_fpr(val_df["label"], val_df["score"], TARGET_FPR)
print("tau (FPR=1%% tren val) = %.4f" % TAU, flush=True)

def report_split(name, d):
    if d is None:
        return None
    y, s = d["label"].values, d["score"].values
    row = {"split": name, "n": len(d), "n_ai": int((y == 1).sum()), "auroc": auroc(y, s),
           "auprc": float(average_precision_score(y, s)) if len(np.unique(y)) == 2 else np.nan}
    for f in [0.001, 0.01, 0.05]:
        row["tpr@fpr%s" % f] = tpr_at_fpr(y, s, f)[0]
    row["tpr@tau"] = float((s[y == 1] >= TAU).mean()) if (y == 1).any() else np.nan
    row["fpr@tau"] = float((s[y == 0] >= TAU).mean()) if (y == 0).any() else np.nan
    return row

rows = [r for r in [report_split("val", val_df), report_split("test_ood_model", ood_df),
                    report_split("test_fp", fp_df)] if r]

# ── ĐÁNH GIÁ ĐỐI-SHORTCUT trên TẬP MATCHED (make_matched_eval.py) ─────
# Đây là con số trả lời phản biện "0.99 của anh chỉ là đếm chữ với độ dài từ".
#
# Tập matched là tập con mà MỌI mô hình chỉ-đặc-trưng-bề-mặt đều về mức ngẫu nhiên —
# đã kiểm bằng hai mô hình để không lập luận vòng tròn (fit trên train, và fit LẠI
# out-of-fold trên chính tập con):
#     val            876 -> 296 dòng    surface 0.9049 -> 0.5027 / 0.5475
#     test_ood_model 3.028 -> 1.206     surface 0.8715 -> 0.5446 / 0.4663
# Nên nếu detector vẫn giữ AUROC cao ở đây thì phần đó KHÔNG đến từ đặc trưng bề mặt.
#
# Sàn để so: GBM 22 đặc trưng bề mặt đạt AUROC 0.8715 trên test_ood_model đầy đủ
# nhưng 0.5446 trên tập matched (surface_baseline.py). Khoảng cách giữa detector và
# sàn đó là phần detector thực sự đóng góp.
#
# Chấm lại trực tiếp trên text của tập matched, KHÔNG join theo chỉ số dòng: tập
# matched chỉ giữ ~1/3 dòng và thứ tự khác, join theo vị trí là sai âm thầm.
# Tra globals() thay vì gọi thẳng tên: nhánh supervised có predict_proba() và
# SMOKE_TEST, nhánh RADAR dùng detector_scores() và SMOKE. Gọi thẳng thì block này
# chạy được ở nhánh này và NameError ở nhánh kia — mà lỗi đó chỉ nổ ở CUỐI buổi
# train, sau khi đã tốn hết GPU-hour.
_score_fn = globals().get("predict_proba") or globals().get("detector_scores")
_smoke = bool(globals().get("SMOKE_TEST") or globals().get("SMOKE") or False)
assert _score_fn is not None, "không tìm thấy hàm chấm điểm (predict_proba/detector_scores)"

for _sp in ("val", "test_ood_model"):
    _mp = DATA_DIR / f"matched_eval_{_sp}_{DATASET_TAG}.parquet"
    if not _mp.exists():
        print(f"  (thiếu {_mp.name} -> bỏ qua đánh giá matched cho {_sp})", flush=True)
        continue
    _m = pd.read_parquet(_mp)
    _m["score"] = _score_fn(_m["text"].tolist())
    _row = report_split(f"{_sp}_matched", _m)
    if _row:
        rows.append(_row)
        print(f"  matched {_sp}: n={len(_m)} AUROC={_row['auroc']:.4f}", flush=True)
    if not _smoke:
        _keep = [c for c in ("sample_id", "paper_id", "label", "text_role",
                             "surface_propensity", "score") if c in _m.columns]
        _m[_keep].to_parquet(OUT_DIR / f"pred_matched_{_sp}.parquet", index=False)
metrics_df = pd.DataFrame(rows).set_index("split")
pd.set_option("display.width", 200, "display.max_columns", 20)
print(metrics_df.round(4).to_string(), flush=True)

val_auc = metrics_df.loc["val", "auroc"] if "val" in metrics_df.index else np.nan
ood_auc = metrics_df.loc["test_ood_model", "auroc"] if "test_ood_model" in metrics_df.index else np.nan
print("=" * 60); print("HEADLINE -- %s" % RUN_NAME); print("=" * 60)
if "test_ood_model" in metrics_df.index:
    print("1) TPR@FPR=1%% OOD-model : %.3f" % metrics_df.loc["test_ood_model", "tpr@fpr0.01"])
if "test_fp" in metrics_df.index:
    print("2) FPR thuc @ tau test_fp: %.4f" % metrics_df.loc["test_fp", "fpr@tau"])
print("3) AUROC gap val->OOD   : %.3f -> %.3f  (delta=%+.3f)" % (val_auc, ood_auc, val_auc - ood_auc), flush=True)

(OUT_DIR / "metrics.json").write_text(json.dumps(
    {"run": RUN_NAME, "model": DETECTOR, "method": "greater", "eval_mode": "inductive",
     "surrogate": SURROGATE, "substitution_level": "word", "tau": TAU, "target_fpr": TARGET_FPR,
     "seed": SEED, "sample_weight_used": False,
     "hparams": {"lambda": LAMBDA_A, "eps": EPS_SCALE, "xi": XI_SCALE,
                 "label_smoothing": LABEL_SMOOTH, "k_percent": K_PERCENT, "top_m": TOP_M,
                 "attack_frac": ATTACK_FRAC, "epochs_run": len(monitor), "best_val_auroc": best,
                 "effective_batch": EFFECTIVE_BATCH, "det_lr": DET_LR, "score_lr": SCORE_LR,
                 "max_len": MAX_LEN},
     "metrics": metrics_df.reset_index().to_dict("records")}, indent=2, default=float))
for nm, d in [("val", val_df), ("test_ood_model", ood_df), ("test_fp", fp_df)]:
    if d is not None:
        d[["paper_id", "label", "ai_degree", "generator_family", "score"]].to_parquet(
            OUT_DIR / ("pred_%s.parquet" % nm), index=False)
print("da luu artifacts ->", OUT_DIR, flush=True)

## PROBE VRAM — chạy **trước** SMOKE (~3-5 phút)

> **TẮT cho `Run All`** — cell dưới đã comment. Bỏ dấu `#` để chạy lại khi cần (sau khi sửa code, đổi model, hoặc nghi OOM/rò VRAM).


Lặp **cùng một batch** qua đúng đường tấn công (`greater_a` → `adversary_loss` → `backward`)
`PROBE_ITERS` vòng rồi in VRAM trên `SUR_DEV`. `SMOKE=1` rút warm-up `M_sur` còn 32 step.

Mục đích: biết đường tấn công còn rò VRAM hay không **trước khi** cam kết ~8 tiếng full run —
lần trước OOM rơi vào giữa epoch 4, tức mất 5.5 giờ mới biết.

| Kết quả | Nghĩa là | Làm gì |
|---|---|---|
| `delta vs base` đi ngang quanh 0 | sạch | chạy SMOKE rồi FULL |
| `delta vs base` leo đều theo vòng | còn leak | **đừng** chạy FULL |
| `alloc` cao nhưng `sau gc` tụt mạnh | leak do reference cycle | hạ `GC_EVERY` 200 → 50 |

`peak` là đỉnh thật của một vòng tấn công — dùng để biết còn bao nhiêu headroom trên 14.56 GiB.

> Chậm quá thì `PROBE_ITERS=40`. Xu hướng vẫn đọc được, chỉ nhiễu hơn.


In [ ]:
# !SMOKE=1 PROBE=1 python train_greater.py


## SMOKE

> **TẮT cho `Run All`** — cell dưới đã comment. Bỏ dấu `#` để chạy lại khi cần (sau khi sửa code, đổi model, hoặc nghi OOM/rò VRAM).


| Gate | Kỳ vọng | Nếu fail |
|---|---|---|
| Chạy hết, không OOM | ✅ | xác nhận log đã giải phóng optimizer warm-up; nếu vẫn fail, đặt attack micro-batch=1, **không** hạ `SUR_MAX_LEN` |
| `doi N tu` > 0 | greedy search thay được từ | bộ lọc POS quá chặt → hạ `MIN_WORD_LEN` hoặc nới `TOP_M` |
| `ASR` > 0 ở vài batch | adversary lừa được detector → vòng đối kháng có tín hiệu | xem trên |
| `L_imp` giảm | `F_θ` đang học đoán importance từ gradient `M_sur` | kiểm warm-up `M_sur` có hội tụ không |
| `AUROC(val)` > 0.5 | detector vẫn học bài chính | — |

> `ASR = 0` **toàn bộ** nghĩa là attack vô hiệu → `L_adv` bão hoà. Kiểm `doi N tu` trước: nếu = 0 thì
> vấn đề nằm ở khâu sinh ứng viên chứ không phải ở detector.

In [ ]:
# !SMOKE=1 python train_greater.py

## FULL RUN

6 epoch (đúng paper), early-stop patience 2 theo AUROC(val) (đúng supervised).

Chi phí trội so với train thường: warm-up `M_sur` 1 epoch (`max_len=512`) + với mỗi batch, greedy
search/pruning trên ~20% mặt AI, mỗi round **1 lần query `M_tar` cho cả lô**.

In [ ]:
!python train_greater.py

In [ ]:
# ============ Bao cao ============
import json, pandas as pd
from pathlib import Path

OUT = Path("artifacts_greater_xlm_roberta_large_seed42")
m = json.loads((OUT / "metrics.json").read_text())
print("=== GREATER (ACL 2025) ===")
print("M_tar %s | M_sur %s | thay o muc: %s" % (m["model"], m["surrogate"], m["substitution_level"]))
print("epoch da chay: %d | best AUROC(val): %.4f"
      % (m["hparams"]["epochs_run"], m["hparams"]["best_val_auroc"]))
print(pd.DataFrame(m["metrics"]).set_index("split").round(4).to_string())

mon = pd.read_csv(OUT / "monitor.csv")
print("\n--- monitor ---")
print(mon.round(4).to_string(index=False))
print("\nASR tang dan = adversary manh len; AUROC(val) van giu = detector khong bi pha.")